In [1]:
import torch
print(torch.cuda.is_available())
print(torch.__version__)
print(torch.cuda.mem_get_info())

True
2.9.1+cu130
(7411335168, 8585216000)


In [2]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM,AutoTokenizer,BitsAndBytesConfig,TrainingArguments
from peft import LoraConfig, get_peft_model,prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig


In [3]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
#dataset loading and inspection
from datasets import load_dataset, concatenate_datasets
pubmed_dataset=load_dataset("qiaojin/PubMedQA", "pqa_labeled", trust_remote_code=True)
medmcqa_dataset=load_dataset("openlifescienceai/medmcqa")

print("PubMedQA:", pubmed_dataset)
print("\nMedMCQA:", medmcqa_dataset)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'qiaojin/PubMedQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


PubMedQA: DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 1000
    })
})

MedMCQA: DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name'],
        num_rows: 182822
    })
    test: Dataset({
        features: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name'],
        num_rows: 6150
    })
    validation: Dataset({
        features: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name'],
        num_rows: 4183
    })
})


In [ ]:
#imports for qlora parameter configuration
from transformers import BitsAndBytesConfig
import torch
bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [ ]:
#import for the llm 
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name="meta-llama/Llama-3.2-3B-Instruct"
tokenizer=AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side="right"

model=AutoModelForCausalLM.from_pretrained(model_name,
                                           quantization_config=bnb_config,
                                           device_map="auto"
                                           )
print(model)
print(f"\nModel loaded! Memory used: {torch.cuda.memory_allocated()/1024**3:.2f} GB")



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNo

In [ ]:
#data formatting for the training
def format_pubmed(example):
    context=" ".join(example['context']['contexts'])
    message=[
        {
            "role":"system",
            "context":"You are a medical expert.Answer questions accurately based on the provided context."  
        },
        {
            "role":"user",
            "context":f"Context: {context}\nQuestion: {example['question']}"
        },
        {
            "role":"assistant",
            "content": f"{example['long_answer']} (Final decision: {example['final_decision']})"
        }
    ]
    return {"text":tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=False)}

def format_medmcqa(example):
    options = {0: example['opa'], 1: example['opb'], 2: example['opc'], 3: example['opd']}
    correct_answer = options[example['cop']]
    
    # clean the explanation
    raw_exp = example['exp'] if example['exp'] else f"The answer is {correct_answer}."
    explanation = clean_explanation(raw_exp)
    
    # if explanation too short after cleaning use correct answer as fallback
    if len(explanation) < 20:
        explanation = f"{correct_answer}."

    messages = [
        {
            "role": "system",
            "content": "You are a medical expert. Answer questions accurately and explain your reasoning."
        },
        {
            "role": "user",
            "content": f"{example['question']}"
        },
        {
            "role": "assistant",
            "content": explanation
        }
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}
print("Formatting PubMedQA...")
formatted_pubmed = pubmed_dataset['train'].map(format_pubmed)

# split BEFORE combining — 900 train, 100 test
pubmed_split = formatted_pubmed.train_test_split(test_size=0.1, seed=42)
pubmed_train = pubmed_split['train']   # 900 examples
test_dataset  = pubmed_split['test']   # 100 examples — kept separate for evaluation
pubmed_train = pubmed_train.select_columns(['text'])

# ── 4. Format MedMCQA ───────────────────────────────────────────
print("Formatting MedMCQA (5000 examples)...")
medmcqa_subset = medmcqa_dataset['train'].shuffle(seed=42).select(range(5000))
formatted_medmcqa = medmcqa_subset.map(format_medmcqa)
formatted_medmcqa = formatted_medmcqa.select_columns(['text'])

# ── 5. Combine only train splits ────────────────────────────────
train_dataset = concatenate_datasets([pubmed_train, formatted_medmcqa])
train_dataset = train_dataset.shuffle(seed=42)

print(f"\nPubMedQA train : {len(pubmed_train)}")
print(f"MedMCQA train  : {len(formatted_medmcqa)}")
print(f"Combined train : {len(train_dataset)}")
print(f"Test set       : {len(test_dataset)} (PubMedQA only, never seen during training)")
    



Formatting PubMedQA...
Formatting MedMCQA (5000 examples)...


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]


PubMedQA train : 900
MedMCQA train  : 5000
Combined train : 5900
Test set       : 100 (PubMedQA only, never seen during training)


In [ ]:
#data formatting for the fixing the training data
import re

def clean_explanation(text):
    if not text:
        return text
    
    # remove Ans. prefix first
    text = re.sub(r'^Ans[\.\s]+\([a-dA-D]\)\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^Ans[\.\s]+(is\s+)?[\'"]?[a-dA-D][\'"]?\s*(i\.e[\.,])?\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^Answer[\s:]+[a-dA-D][\s\.,]*', '', text, flags=re.IGNORECASE)
    
    # remove ONLY the reference citation inline, not everything after it
    # pattern: Ref: Harrison's 19th ed. P 640* → remove just this part
    text = re.sub(r'Ref[:\s]+[^*\n]+[\*\s]*', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'Refer[:\s]+[^*\n]+[\*\s]*', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'Reference[:\s]+[^*\n]+[\*\s]*', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\(Ref[^)]*\)', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'pg[\s\.]?\d+', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'page[\s\.]?\d+', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\d+[a-z]+\s*edition', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\d+[a-z]+\/e', ' ', text, flags=re.IGNORECASE)
    
    # clean dangling punctuation at start
    text = re.sub(r'^[\s,\.\-:]+', '', text)
    
    # clean up extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# test
for i in range(5):
    exp = medmcqa_dataset['train'][i]['exp']
    if exp:
        cleaned = clean_explanation(exp)
        print(f"BEFORE: {exp[:250]}")
        print(f"AFTER:  {cleaned[:250]}")
        print("="*60)

BEFORE: Chronic urethral obstruction because of urinary calculi, prostatic hyperophy, tumors, normal pregnancy, tumors, uterine prolapse or functional disorders cause hydronephrosis which by definition is used to describe dilatation of renal pelvis and calcu
AFTER:  Chronic urethral obstruction because of urinary calculi, prostatic hyperophy, tumors, normal pregnancy, tumors, uterine prolapse or functional disorders cause hydronephrosis which by definition is used to describe dilatation of renal pelvis and calcu
BEFORE: Ans. (c) Vitamin B12 Ref: Harrison's 19th ed. P 640* Vitamin B12 (Cobalamin) is synthesized solely by microorganisms.* In humans, the only source for humans is food of animal origin, e.g., meat, fish, and dairy products.* Vegetables, fruits, and othe
AFTER:  Vitamin B12 Vitamin B12 (Cobalamin) is synthesized solely by microorganisms.* In humans, the only source for humans is food of animal origin, e.g., meat, fish, and dairy products.* Vegetables, fruits, and other food

In [10]:
print(formatted_medmcqa[2])

{'text': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 13 Apr 2026\n\nYou are a medical expert. Answer questions accurately and explain your reasoning.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nInvestigation of choice to diagnose congenital malformations<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nMRI is the investigation of choice to diagnose congenital malformaiton.<|eot_id|>'}


In [ ]:
#lora parameter configuration
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# prepare model for training
model = prepare_model_for_kbit_training(model)

# configure LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# inject LoRA adapters into model
model = get_peft_model(model, lora_config)

# see how many parameters we're actually training
model.print_trainable_parameters()

trainable params: 9,175,040 || all params: 3,221,924,864 || trainable%: 0.2848


In [12]:
# check average token length of combined dataset
sample_lengths = []
for i in range(0, min(500, len(train_dataset))):
    tokens = tokenizer(train_dataset[i]['text'], return_tensors="pt")
    sample_lengths.append(tokens['input_ids'].shape[1])

import numpy as np
print(f"Average length : {np.mean(sample_lengths):.0f} tokens")
print(f"Max length     : {np.max(sample_lengths)} tokens")
print(f"95th percentile: {np.percentile(sample_lengths, 95):.0f} tokens")

Average length : 159 tokens
Max length     : 1335 tokens
95th percentile: 387 tokens


In [13]:
# filter out examples longer than 512 tokens
def is_short_enough(example):
    tokens = tokenizer(example['text'], return_tensors="pt")
    return tokens['input_ids'].shape[1] <= 512

print("Filtering long examples...")
train_dataset_filtered = train_dataset.filter(is_short_enough)

print(f"Before filtering: {len(train_dataset)}")
print(f"After filtering : {len(train_dataset_filtered)}")

Filtering long examples...


Filter:   0%|          | 0/5900 [00:00<?, ? examples/s]

Before filtering: 5900
After filtering : 5790


In [ ]:
#activate training mode for the model
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir="./medllm-output-v2",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=50,
    save_steps=500,
    dataset_text_field="text",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset_filtered,
    args=sft_config,
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/5790 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5790 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


Step,Training Loss
50,2.069100
100,1.784500
150,1.726100
200,1.699000
250,1.662500
300,1.556300
350,1.571100
400,1.567300
450,1.507100
500,1.599900


TrainOutput(global_step=4344, training_loss=1.4337254150137717, metrics={'train_runtime': 10435.5822, 'train_samples_per_second': 1.664, 'train_steps_per_second': 0.416, 'total_flos': 4.191325485176218e+16, 'train_loss': 1.4337254150137717})

In [ ]:
#saving the model adapters
model.save_pretrained("./medllm-adapter-v2")
tokenizer.save_pretrained("./medllm-adapter-v2")
print("Adapter v2 saved!")

Adapter v2 saved!


In [14]:
# disable gradient checkpointing for inference
model.config.use_cache = True
model.gradient_checkpointing_disable()

messages = [
    {"role": "system", "content": "You are a medical expert."},
    {"role": "user", "content": "What is diabetes?"}
]

input_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.eos_token_id
    )

new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))

Diabetes mellitus (DM) is an important public health problem worldwide, characterized by hyperglycemia due to impaired insulin secretion and/or increased glucose production in the liver. The pathogenesis of DM involves genetic predisposition as well as environmental factors such as diet, obesity, physical activity level, smoking status, age, family history, ethnicity, pregnancy, infections, autoimmune diseases, drugs, etc., which may lead to beta-cell destruction or dysfunction leading to chronic hyperinsulinemic euglycaemic state known as type-1 diabetes mellitus; whereas non-insuline dependent diabetic syndrome occurs when there is relative deficiency of endogenous insulin with compensatory increase in exogenously administered insulin resulting from resistance to its action on peripheral tissues.

(


In [ ]:
# score checker 
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bert_score_fn
import nltk
import numpy as np
import torch

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
!pip install bert-score -q

# ── Setup ────────────────────────────────────────────────────────
rouge_scorer_obj = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
smoothing = SmoothingFunction().method1

model.config.use_cache = True
model.gradient_checkpointing_disable()

def generate_prediction(question, context=None, max_new_tokens=100):
    if context:
        messages = [
            {"role": "system", "content": "You are a medical expert. Answer questions accurately based on the provided context."},
            {"role": "user", "content": f"Context: {context}\n\nQuestion: {question}"}
        ]
    else:
        messages = [
            {"role": "system", "content": "You are a medical expert. Answer questions accurately and explain your reasoning."},
            {"role": "user", "content": question}
        ]

    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            early_stopping=True,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

# ── PubMedQA Evaluation ──────────────────────────────────────────
pubmed_rouge = []
pubmed_bleu  = []
pubmed_preds = []
pubmed_refs  = []

print("="*60)
print("Evaluating PubMedQA (100 examples)...")
print("="*60)

for i in range(len(test_dataset)):
    ref      = test_dataset[i]['long_answer']
    question = test_dataset[i]['question']
    context  = " ".join(test_dataset[i]['context']['contexts'])

    prediction = generate_prediction(question, context)

    # ROUGE-L
    r = rouge_scorer_obj.score(ref, prediction)
    pubmed_rouge.append(r['rougeL'].fmeasure)

    # BLEU
    ref_tok  = nltk.word_tokenize(ref.lower())
    pred_tok = nltk.word_tokenize(prediction.lower())
    pubmed_bleu.append(sentence_bleu([ref_tok], pred_tok, smoothing_function=smoothing))

    # store for BERTScore
    pubmed_preds.append(prediction)
    pubmed_refs.append(ref)

    if i % 10 == 0:
        print(f"  Progress: {i:>3}/100 | ROUGE-L: {np.mean(pubmed_rouge):.4f} | BLEU: {np.mean(pubmed_bleu):.4f}")

print(f"\nPubMedQA → ROUGE-L: {np.mean(pubmed_rouge):.4f} | BLEU: {np.mean(pubmed_bleu):.4f}")

# ── MedMCQA Evaluation ───────────────────────────────────────────
medmcqa_rouge = []
medmcqa_bleu  = []
medmcqa_preds = []
medmcqa_refs  = []

medmcqa_eval = medmcqa_dataset['validation'].shuffle(seed=42).select(range(100))

print("\n" + "="*60)
print("Evaluating MedMCQA (100 examples)...")
print("="*60)

for i in range(100):
    example  = medmcqa_eval[i]
    question = example['question']
    raw_ref  = example['exp'] if example['exp'] else ""
    ref      = clean_explanation(raw_ref)

    # skip if reference too short after cleaning
    if len(ref) < 20:
        continue

    prediction = generate_prediction(question)

    # ROUGE-L
    r = rouge_scorer_obj.score(ref, prediction)
    medmcqa_rouge.append(r['rougeL'].fmeasure)

    # BLEU
    ref_tok  = nltk.word_tokenize(ref.lower())
    pred_tok = nltk.word_tokenize(prediction.lower())
    medmcqa_bleu.append(sentence_bleu([ref_tok], pred_tok, smoothing_function=smoothing))

    # store for BERTScore
    medmcqa_preds.append(prediction)
    medmcqa_refs.append(ref)

    if i % 10 == 0:
        print(f"  Progress: {i:>3}/100 | ROUGE-L: {np.mean(medmcqa_rouge):.4f} | BLEU: {np.mean(medmcqa_bleu):.4f}")

print(f"\nMedMCQA  → ROUGE-L: {np.mean(medmcqa_rouge):.4f} | BLEU: {np.mean(medmcqa_bleu):.4f}")

# ── BERTScore (batch) ────────────────────────────────────────────
print("\n" + "="*60)
print("Calculating BERTScore...")
print("="*60)

print("  BERTScore for PubMedQA...")
_, _, F1_pubmed = bert_score_fn(
    pubmed_preds,
    pubmed_refs,
    lang="en",
    device="cuda",
    batch_size=8,
    verbose=False
)

print("  BERTScore for MedMCQA...")
_, _, F1_medmcqa = bert_score_fn(
    medmcqa_preds,
    medmcqa_refs,
    lang="en",
    device="cuda",
    batch_size=8,
    verbose=False
)

# ── Final Summary Table ──────────────────────────────────────────
combined_rouge = np.mean(pubmed_rouge + medmcqa_rouge)
combined_bleu  = np.mean(pubmed_bleu  + medmcqa_bleu)
combined_bert  = (F1_pubmed.mean().item() + F1_medmcqa.mean().item()) / 2

print("\n" + "="*60)
print("           FINAL EVALUATION SUMMARY")
print("="*60)
print(f"{'Dataset':<12} {'ROUGE-L':<12} {'BLEU':<12} {'BERTScore':<12}")
print(f"{'-'*48}")
print(f"{'PubMedQA':<12} {np.mean(pubmed_rouge):<12.4f} {np.mean(pubmed_bleu):<12.4f} {F1_pubmed.mean().item():<12.4f}")
print(f"{'MedMCQA':<12} {np.mean(medmcqa_rouge):<12.4f} {np.mean(medmcqa_bleu):<12.4f} {F1_medmcqa.mean().item():<12.4f}")
print(f"{'-'*48}")
print(f"{'Combined':<12} {combined_rouge:<12.4f} {combined_bleu:<12.4f} {combined_bert:<12.4f}")
print("="*60)
print("\n" + "="*60)
print("           FINAL EVALUATION SUMMARY")
print("="*60)
print(f"{'Dataset':<12} {'ROUGE-L':<12} {'BLEU':<12} {'BERTScore':<12}")
print(f"{'-'*48}")
print(f"{'PubMedQA':<12} {np.mean(pubmed_rouge):<12.4f} {np.mean(pubmed_bleu):<12.4f} {F1_pubmed.mean().item():<12.4f}")
print(f"{'MedMCQA':<12} {np.mean(medmcqa_rouge):<12.4f} {np.mean(medmcqa_bleu):<12.4f} {F1_medmcqa.mean().item():<12.4f}")
print(f"{'-'*48}")
combined_rouge = np.mean(pubmed_rouge + medmcqa_rouge)
combined_bleu  = np.mean(pubmed_bleu  + medmcqa_bleu)
combined_bert  = (F1_pubmed.mean().item() + F1_medmcqa.mean().item()) / 2
print(f"{'Combined':<12} {combined_rouge:<12.4f} {combined_bleu:<12.4f} {combined_bert:<12.4f}")
print("="*60)

Evaluating PubMedQA (100 examples)...
  Progress:   0/100 | ROUGE-L: 0.1085 | BLEU: 0.0091
  Progress:  10/100 | ROUGE-L: 0.0792 | BLEU: 0.0050
  Progress:  20/100 | ROUGE-L: 0.0937 | BLEU: 0.0060
  Progress:  30/100 | ROUGE-L: 0.0857 | BLEU: 0.0057
  Progress:  40/100 | ROUGE-L: 0.0825 | BLEU: 0.0057
  Progress:  50/100 | ROUGE-L: 0.0791 | BLEU: 0.0052
  Progress:  60/100 | ROUGE-L: 0.0768 | BLEU: 0.0052
  Progress:  70/100 | ROUGE-L: 0.0781 | BLEU: 0.0057
  Progress:  80/100 | ROUGE-L: 0.0818 | BLEU: 0.0058
  Progress:  90/100 | ROUGE-L: 0.0823 | BLEU: 0.0056

PubMedQA → ROUGE-L: 0.0812 | BLEU: 0.0055

Evaluating MedMCQA (100 examples)...
  Progress:  30/100 | ROUGE-L: 0.0789 | BLEU: 0.0035
  Progress:  50/100 | ROUGE-L: 0.0739 | BLEU: 0.0036
  Progress:  70/100 | ROUGE-L: 0.0775 | BLEU: 0.0033
  Progress:  80/100 | ROUGE-L: 0.0780 | BLEU: 0.0031
  Progress:  90/100 | ROUGE-L: 0.0792 | BLEU: 0.0031

MedMCQA  → ROUGE-L: 0.0780 | BLEU: 0.0030

Calculating BERTScore...
  BERTScore for P

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  BERTScore for MedMCQA...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



           FINAL EVALUATION SUMMARY
Dataset      ROUGE-L      BLEU         BERTScore   
------------------------------------------------
PubMedQA     0.0812       0.0055       0.8318      
MedMCQA      0.0780       0.0030       0.8177      
------------------------------------------------
Combined     0.0801       0.0047       0.8248      

           FINAL EVALUATION SUMMARY
Dataset      ROUGE-L      BLEU         BERTScore   
------------------------------------------------
PubMedQA     0.0812       0.0055       0.8318      
MedMCQA      0.0780       0.0030       0.8177      
------------------------------------------------
Combined     0.0801       0.0047       0.8248      


In [ ]:
#if you have the weight files locally, you can load and test the model like this:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

model_name = "meta-llama/Llama-3.2-3B-Instruct"
adapter_path = "./medllm-adapter-v2"  # your saved adapter

# 4-bit config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# load base model
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# load your LoRA adapter on top
print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, adapter_path)
tokenizer = AutoTokenizer.from_pretrained(adapter_path)
tokenizer.pad_token = tokenizer.eos_token

# set inference mode
model.config.use_cache = True
model.gradient_checkpointing_disable()

print(f"Model loaded! VRAM used: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded with adapter!


In [ ]:
#testing then model wqith a general medical question (no context needed)
model.config.use_cache = True
model.gradient_checkpointing_disable()

# test with a general medical question (no context needed)
messages = [
    {"role": "system", "content": "You are a medical expert. Answer questions accurately and explain your reasoning clearly."},
    {"role": "user", "content": "What are the symptoms of myocardial infarction?"}
]

input_text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.eos_token_id,
    )

new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))

The answer is Chest pain, cold sweats & anxiety..


In [31]:
test_questions = [
    "What is the treatment for Type 2 diabetes?",
    "What causes pneumonia?",
    "What are the signs of appendicitis?"
]

for question in test_questions:
    messages = [
        {"role": "system", "content": "You are a medical chatbot. Answer questions in depth and explain your reasoning clearly."},
        {"role": "user", "content": question}
    ]
    
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)
    print(f"Q: {question}")
    print(f"A: {answer}")
    print("="*60)

Q: What is the treatment for Type 2 diabetes?
A: The answer is Lifestyle modification, oral hypoglycemic drugs & insulin therapy..
Q: What causes pneumonia?
A: Pneumonia is an inflammation of the lung tissue, usually caused by infection with bacteria or viruses.Pneumococcus (Streptococcus pneumonae) is one common cause.Bacterial infections can also be triggered by viral respiratory tract illnesses such as influenza.Among other bacterial agents that may infect lungs include Klebsiella pneumoniae, Legionnaires' disease-causing legionellabacteria, Haemophilus influenzae type b, Mycoplasma pneumoniae, Staphylococcus aureus, Streptomyces pyogenes, Corynebacterium diphtheriae, Chlamydophila pneumoniae. Viral pathogens causing acute bronchopneumonic illness typically belong to families poxviridae, heparoviruses, adenovirus family, herpesvirus, parainfluenza virus, coronavirus, rhinovirus, enterovirus and picornavirus.Fungal organisms including Aspergillus species, Histoplasmosis fungus, Crypt

In [36]:
test_questions = [
    "What is the treatment for Type 2 diabetes?",
    "What are the symptoms of pneumonia?",
    "What are the signs of appendicitis?"
]

for question in test_questions:
    messages = [
        {"role": "system", "content": "You are a medical chatbot. Answer questions in depth and explain your reasoning clearly."},
        {"role": "user", "content": question}
    ]
    
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=True,
            temperature=0.3,
            repetition_penalty=1.1,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)
    print(f"Q: {question}")
    print(f"A: {answer}")
    print("="*60)

Q: What is the treatment for Type 2 diabetes?
A: Type 2 Diabetes Mellitus (T2DM) It is a chronic metabolic disorder characterized by insulin resistance, impaired insulin secretion, and hyperglycemia. The pathogenesis of T2DM involves multiple factors including genetic predisposition, obesity, sedentary lifestyle, diet high in carbohydrate and fat, and low in fiber, and increased oxidative stress. Treatment of T1DM includes insulin therapy. Treatment for T2 DM include lifestyle modification such as dietary changes, exercise, weight loss and oral hypoglycaemic agents like metformin, sulfonylureas, thiazolidinediones, alpha-glucosidase inhibitors, DPP-4 inhibitors, GLP-1 receptor agonists and SGLT-2 inhibitors.
Q: What are the symptoms of pneumonia?
A: Fever, chills, cough with yellow sputum, chest pain, breathlessness, fatigue, headache, general malaise, weight loss, sweating, shivering, anorexia, nausea, vomiting, abdominal pain, fever, chILLS, cough, chest tightness, shortness of breat

In [37]:
import os
files = os.listdir("./medllm-adapter-v2")
print(files)

['adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'README.md', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json']


In [41]:
import gradio as gr
import torch

# make sure model is in inference mode
model.config.use_cache = True
model.gradient_checkpointing_disable()

def medical_answer(question):
    if not question.strip():
        return "Please enter a medical question."
    
    messages = [
        {
            "role": "system",
            "content": "You are a medical chatbot. Answer questions in depth and explain your reasoning clearly.."
        },
        {
            "role": "user",
            "content": question
        }
    ]
    
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id
        )
    
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return answer

# ── Build Gradio UI ─────────────────────────────────────────────
with gr.Blocks(title="MedLLM — Medical QA") as demo:
    
    gr.Markdown("""
    # 🏥 MedLLM — Medical Question Answering
    Fine-tuned LLaMA 3.2 3B on PubMedQA + MedMCQA using QLoRA
    
    > ⚠️ For educational purposes only. Not for clinical use.
    """)
    
    with gr.Row():
        with gr.Column():
            question_input = gr.Textbox(
                label="Medical Question",
                placeholder="e.g. What are the symptoms of myocardial infarction?",
                lines=3
            )
            submit_btn = gr.Button("Get Answer", variant="primary")
            
            gr.Examples(
                examples=[
                    ["What are the symptoms of myocardial infarction?"],
                    ["What causes pneumonia?"],
                    ["What is the treatment for Type 2 diabetes?"],
                    ["What are the signs of appendicitis?"],
                    ["What is the difference between Type 1 and Type 2 diabetes?"],
                ],
                inputs=question_input
            )
    
    with gr.Row():
        answer_output = gr.Textbox(
            label="Model Answer",
            lines=8,
            interactive=False
        )
    
    submit_btn.click(
        fn=medical_answer,
        inputs=question_input,
        outputs=answer_output
    )
    
    question_input.submit(
        fn=medical_answer,
        inputs=question_input,
        outputs=answer_output
    )

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://21e9987a137c63254f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
